In [1]:
import tkinter as tk
from tkinter import ttk, messagebox

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

# Visualization style
sns.set_theme(style="whitegrid")

# Main window
root = tk.Tk()
root.title("Unit Converter - Data Visualization")
root.geometry("1150x750")
root.configure(bg="#EAF2F8")
root.minsize(950, 650)

# Conversion categories
units = {
    "Length": ["Kilometres", "Metres"],
    "Weight": ["Kilograms", "Grams"],
    "Temperature": ["Celsius", "Fahrenheit"]
}

# Conversion history
history = []

# Main title
title = tk.Label(
    root,
    text="UNIT CONVERTER WITH DATA VISUALIZATION",
    font=("Arial", 20, "bold"),
    bg="#1F4E79",
    fg="white",
    pady=15
)
title.pack(fill="x")

# Main container
main_frame = tk.Frame(root, bg="#EAF2F8")
main_frame.pack(fill="both", expand=True, padx=15, pady=15)

# Left control panel
control_frame = tk.Frame(
    main_frame,
    bg="white",
    padx=20,
    pady=20
)
control_frame.pack(side="left", fill="y", padx=10)

tk.Label(
    control_frame,
    text="UNIT CONVERTER",
    font=("Arial", 16, "bold"),
    bg="white",
    fg="#1F4E79"
).pack(pady=15)

# Category selection
tk.Label(
    control_frame,
    text="Select Category",
    font=("Arial", 11, "bold"),
    bg="white"
).pack(anchor="w")

category = ttk.Combobox(
    control_frame,
    values=list(units.keys()),
    state="readonly",
    font=("Arial", 11),
    width=22
)
category.pack(pady=8)
category.set("Length")

# From unit
tk.Label(
    control_frame,
    text="From Unit",
    font=("Arial", 11, "bold"),
    bg="white"
).pack(anchor="w")

from_unit = ttk.Combobox(
    control_frame,
    state="readonly",
    font=("Arial", 11),
    width=22
)
from_unit.pack(pady=8)

# To unit
tk.Label(
    control_frame,
    text="To Unit",
    font=("Arial", 11, "bold"),
    bg="white"
).pack(anchor="w")

to_unit = ttk.Combobox(
    control_frame,
    state="readonly",
    font=("Arial", 11),
    width=22
)
to_unit.pack(pady=8)

# Input value
tk.Label(
    control_frame,
    text="Enter Value",
    font=("Arial", 11, "bold"),
    bg="white"
).pack(anchor="w")

value_entry = tk.Entry(
    control_frame,
    font=("Arial", 12),
    justify="center",
    width=24
)
value_entry.pack(pady=10)

# Result label
result_label = tk.Label(
    control_frame,
    text="Result will appear here",
    font=("Arial", 12, "bold"),
    fg="#16803C",
    bg="white",
    wraplength=220
)
result_label.pack(pady=15)

# Update units
def update_units(event=None):
    options = units[category.get()]

    from_unit["values"] = options
    to_unit["values"] = options

    from_unit.current(0)
    to_unit.current(1)

category.bind("<<ComboboxSelected>>", update_units)
update_units()

# Conversion logic
def convert_value(value, selected_category, source, target):

    if source == target:
        return value

    if selected_category == "Length":
        if source == "Kilometres":
            return value * 1000
        return value / 1000

    elif selected_category == "Weight":
        if source == "Kilograms":
            return value * 1000
        return value / 1000

    elif selected_category == "Temperature":
        if source == "Celsius":
            return (value * 9 / 5) + 32
        return (value - 32) * 5 / 9


# Visualization dashboard
chart_frame = tk.Frame(
    main_frame,
    bg="white",
    padx=10,
    pady=10
)
chart_frame.pack(side="right", fill="both", expand=True)

chart_title = tk.Label(
    chart_frame,
    text="DATA VISUALIZATION DASHBOARD",
    font=("Arial", 16, "bold"),
    bg="white",
    fg="#1F4E79"
)
chart_title.pack(pady=5)

# Create four charts
figure = Figure(figsize=(8, 5.8), dpi=100)

ax1 = figure.add_subplot(221)
ax2 = figure.add_subplot(222)
ax3 = figure.add_subplot(223)
ax4 = figure.add_subplot(224)

figure.tight_layout(pad=3)

canvas = FigureCanvasTkAgg(
    figure,
    master=chart_frame
)
canvas.get_tk_widget().pack(
    fill="both",
    expand=True
)


# Update all visualizations
def update_visualizations(value, result, category_name,
                          source, target):

    # Generate sample data around input
    if category_name == "Temperature":
        sample_values = [
            value + x for x in range(-10, 11)
        ]
    else:
        step = max(abs(value) * 0.1, 1)
        sample_values = [
            value + (x * step)
            for x in range(-10, 11)
        ]

    converted_values = [
        convert_value(v, category_name, source, target)
        for v in sample_values
    ]

    df = pd.DataFrame({
        "Input": sample_values,
        "Converted": converted_values
    })

    # Clear previous charts
    for ax in [ax1, ax2, ax3, ax4]:
        ax.clear()

    # 1. Bar Chart
    sns.barplot(
        x=["Input", "Converted"],
        y=[value, result],
        hue=["Input", "Converted"],
        palette=["#3498DB", "#2ECC71"],
        legend=False,
        ax=ax1
    )

    ax1.set_title("Bar Chart: Value Comparison")
    ax1.set_ylabel("Value")

    # 2. Line Chart
    sns.lineplot(
        data=df,
        x="Input",
        y="Converted",
        marker="o",
        color="#E67E22",
        ax=ax2
    )

    ax2.set_title("Line Chart: Conversion Trend")
    ax2.set_ylabel(target)

    # 3. Scatter Plot
    sns.scatterplot(
        data=df,
        x="Input",
        y="Converted",
        color="#8E44AD",
        s=45,
        ax=ax3
    )

    ax3.set_title("Scatter Plot: Input vs Output")

    # 4. Histogram
    sns.histplot(
        data=df,
        x="Converted",
        bins=10,
        color="#16A085",
        ax=ax4
    )

    ax4.axvline(
        result,
        color="red",
        linestyle="--",
        label="Current Result"
    )

    ax4.legend()
    ax4.set_title("Histogram: Output Distribution")

    figure.tight_layout(pad=2.5)
    canvas.draw()


# Convert button function
def convert():

    try:
        value = float(value_entry.get())

        selected_category = category.get()
        source = from_unit.get()
        target = to_unit.get()

        if not source or not target:
            messagebox.showerror(
                "Selection Error",
                "Please select both units."
            )
            return

        result = convert_value(
            value,
            selected_category,
            source,
            target
        )

        result_label.config(
            text=f"Result: {result:.2f} {target}"
        )

        history.append({
            "Category": selected_category,
            "Input": value,
            "From": source,
            "Output": result,
            "To": target
        })

        update_visualizations(
            value,
            result,
            selected_category,
            source,
            target
        )

    except ValueError:
        messagebox.showerror(
            "Invalid Input",
            "Please enter a valid number!"
        )


# Convert button
convert_button = tk.Button(
    control_frame,
    text="CONVERT",
    command=convert,
    font=("Arial", 12, "bold"),
    bg="#1F75CB",
    fg="white",
    width=20,
    pady=8
)
convert_button.pack(pady=10)


# Clear function
def clear():

    value_entry.delete(0, tk.END)
    result_label.config(
        text="Result will appear here"
    )

    for ax in [ax1, ax2, ax3, ax4]:
        ax.clear()

    ax1.set_title("Bar Chart")
    ax2.set_title("Line Chart")
    ax3.set_title("Scatter Plot")
    ax4.set_title("Histogram")

    figure.tight_layout(pad=2.5)
    canvas.draw()


# Clear button
clear_button = tk.Button(
    control_frame,
    text="CLEAR",
    command=clear,
    font=("Arial", 11, "bold"),
    bg="#E67E22",
    fg="white",
    width=20,
    pady=8
)
clear_button.pack(pady=8)


# Display conversion history
def show_history():

    if not history:
        messagebox.showinfo(
            "History",
            "No conversion history available."
        )
        return

    history_window = tk.Toplevel(root)
    history_window.title("Conversion History")
    history_window.geometry("750x350")

    columns = (
        "Category", "Input", "From",
        "Output", "To"
    )

    tree = ttk.Treeview(
        history_window,
        columns=columns,
        show="headings"
    )

    for col in columns:
        tree.heading(col, text=col)
        tree.column(col, width=130)

    for item in history:
        tree.insert(
            "",
            tk.END,
            values=(
                item["Category"],
                item["Input"],
                item["From"],
                round(item["Output"], 2),
                item["To"]
            )
        )

    tree.pack(fill="both", expand=True)


history_button = tk.Button(
    control_frame,
    text="VIEW HISTORY",
    command=show_history,
    font=("Arial", 11, "bold"),
    bg="#16803C",
    fg="white",
    width=20,
    pady=8
)
history_button.pack(pady=8)

# Footer
footer = tk.Label(
    root,
    text="Developed using Python | Tkinter | Pandas | Matplotlib | Seaborn",
    font=("Arial", 9),
    bg="#EAF2F8",
    fg="#555555"
)
footer.pack(pady=8)

# Run application
root.mainloop()